# V1

In [1]:
def print_polynomial(f):
    result = ""
    for i in range(len(f)):
        coefficient = str(f[i])
        if coefficient == "0":
            continue
        if i == 0:
            result += f"{coefficient} + "
        elif i == 1:
            if coefficient == "1":
                coefficient = ""
            result += f"{coefficient}x + "
        else:
            if coefficient == "1":
                coefficient = ""
            result += f"{coefficient}x^{i} + "
    print(result.rstrip(" + ")) 

def print_vector(a):
    print()
    for polynomial in a:
        print_polynomial(polynomial)
    print()

In [2]:
def add_polynomials(f, g, modulus):
    result_length = max(len(f), len(g))
    result = [0] * result_length
    for i in range(result_length):
        coefficient_f = f[i] if i < len(f) else 0
        coefficient_g = g[i] if i < len(g) else 0
        result[i] = (coefficient_f + coefficient_g) % modulus
    return result

def subtract_polynomials(f, g, modulus):
    result_length = max(len(f), len(g))
    result = [0] * result_length
    for i in range(result_length):
        coefficient_f = f[i] if i < len(f) else 0
        coefficient_g = g[i] if i < len(g) else 0
        result[i] = (coefficient_f - coefficient_g) % modulus
    return result

def multiply_polynomials(f, g, modulus):
    result_length = len(f) + len(g) - 1
    result = [0] * result_length
    for i in range(len(f)):
        for j in range(len(g)):
            result[i+j] = (result[i+j] + f[i] * g[j]) % modulus
    return result

In [3]:
f = [5, 0, 4, 3]
g = [6, 3, 2]

print_polynomial(f)
print_polynomial(g)

h = add_polynomials(f, g, 7)
print_polynomial(h)

i = subtract_polynomials(f, g, 7)
print_polynomial(i)

j = multiply_polynomials(f, g, 7)
print_polynomial(j)

5 + 4x^2 + 3x^3
6 + 3x + 2x^2
4 + 3x + 6x^2 + 3x^3
6 + 4x + 2x^2 + 3x^3
2 + x + 6x^2 + 2x^3 + 3x^4 + 6x^5


In [4]:
def ring_multiply_polynomials(f, g, modulus, n):
    result = multiply_polynomials(f, g, modulus)
    if len(result) > n:
        for i in range(n, len(result)):
            result[i%n] = (result[i%n] - result[i]) % modulus
        result = result[:n]
    return result

In [5]:
f = [32,0,17,22]
g = [11,7,19,1]
h = ring_multiply_polynomials(f, g, 41, 4)
print_polynomial(h)

39 + 35x + 35x^2 + 24x^3


In [6]:
f = [23,0,11,7]
g = [40,5,16,0]
print(add_polynomials(f, g, 41))
print(subtract_polynomials(f, g, 41))
print(ring_multiply_polynomials(f, g, 41, 4))


[22, 5, 27, 7]
[24, 36, 36, 7]
[12, 3, 29, 7]


In [7]:
def add_vectors(a, b, modulus, k):
    result = []
    for i in range(k):
        result.append(add_polynomials(a[i], b[i], modulus))
    return result

def subtract_vectors(a, b, modulus, k):
    result = []
    for i in range(k):
        result.append(subtract_polynomials(a[i], b[i], modulus))
    return result

def multiply_vectors(a, b, modulus, k, n):
    result = []
    for i in range(k):
        multiplied = ring_multiply_polynomials(a[i], b[i], modulus, n)
        result = add_polynomials(result, multiplied, modulus)
    return result

In [8]:
a = [[93,51,34,54],[27,87,81,6],[112,15,46,122]]
b = [[40,78,1,119],[11,31,57,90],[108,72,47,14]]

print_vector(add_vectors(a, b, 137, 3))
print_vector(subtract_vectors(a, b, 137, 3))

print_polynomial(multiply_vectors(a, b, 137, 3, 4))



133 + 129x + 35x^2 + 36x^3
38 + 118x + x^2 + 96x^3
83 + 87x + 93x^2 + 136x^3


53 + 110x + 33x^2 + 72x^3
16 + 56x + 24x^2 + 53x^3
4 + 80x + 136x^2 + 108x^3

93 + 59x + 44x^2 + 132x^3


In [9]:
def mod_s(r, q):
    r = r % q

    if q % 2 == 0:
        if r <= q // 2:
            result = r
        else:
            result = r - q
    else:
        if r <= (q - 1) // 2:
            result = r
        else:
            result = r - q

    return result


In [10]:
print(mod_s(9+15, 17))
print(mod_s(9-15, 17))
print(mod_s(9*15, 17))
print()
print(mod_s(9+15, 18))
print(mod_s(9-15, 18))
print(mod_s(9*15, 18))

7
-6
-1

6
-6
9


In [11]:
def integer_size(r, q):
    return abs(mod_s(r, q))

def polynomial_size(f, q):
    return max([integer_size(coefficient, q) for coefficient in f])

def vector_size(a, q):
    return max([polynomial_size(polynomial, q) for polynomial in a])

In [12]:
print(integer_size(7, 19))
print(integer_size(18, 19))
print(polynomial_size([1, 12, 0, 3, 0, 18], 19))

7
1
7


# V2

## V2a - simplified PKE scheme

In [13]:
def round_q(x, q):
    x_prime = mod_s(x, q)
    if -int(q/4) <= x_prime <= int(q/4):
        return 0
    else:
        return 1
    
def round_polynomial(f, q):
    return [round_q(coefficient, q) for coefficient in f]

In [14]:
print_polynomial(round_polynomial([3000, 1500, 2010, 37], 3329))

x + x^2


In [15]:
def multiply_square_matrix_by_vector(A, b, modulus, k, n):
    result = [[] for _ in range(k)]

    for row_A in range(k):
        result[row_A] = multiply_vectors(A[row_A], b, modulus, k, n)

    return result

In [16]:
q = 137
n = 4
k = 2
eta1 = 2
eta2 = 2

A = [[[21,57,78,43],[126,122,19,125]],
     [[111,9,63,33],[105,61,71,64]]]

s = [[1,2,-1,2],[0,-1,0,2]]
e = [[1,0,-1,1],[0,-1,1,0]]

As = multiply_square_matrix_by_vector(A, s, q, k, n)
t = add_vectors(As, e, q, k)

print_vector(t)


55 + 96x + 123x^2 + 7x^3
32 + 27x + 127x^2 + 100x^3



In [17]:
def bitstring_to_polynomial(m, n):
    f = [0] * n
    for i in range(n):
        if m[i] == "1":
            f[i] = 1
    return f

def polynomial_to_bitstring(f, n):
    m = ""
    for i in range(n):
        if f[i] == 1:
            m += "1"
        else:
            m += "0"
    return m

In [18]:
def multiply_polynomial_by_scalar(f, scalar, modulus):
    return [(coefficient * scalar) % modulus for coefficient in f]

In [19]:
def matrix_transpose(A, k, n):
    result = [[0] * k for _ in range(n)]
    for i in range(k):
        for j in range(n):
            result[j][i] = A[i][j]
    return result

In [20]:
plaintext = "0111"
m = bitstring_to_polynomial(plaintext, n)

r = [[-2,2,1,-1],[-1,1,1,0]]

e1 = [[1,0,-2,1],[-1,2,-2,1]]

e2 = [2,2,-1,1]

AT = matrix_transpose(A, k, k)

ATr = multiply_square_matrix_by_vector(AT, r, q, k, n)

u = add_vectors(ATr, e1, q, k)

print_vector(u)

tTr = multiply_vectors(t, r, q, k, n)

half_qm = multiply_polynomial_by_scalar(m, (q+1)//2, q)

tTr_e2 = add_polynomials(tTr, e2, q)

v = add_polynomials(tTr_e2, half_qm, q)

print_polynomial(v)


56 + 32x + 77x^2 + 9x^3
45 + 21x + 2x^2 + 127x^3

3 + 10x + 8x^2 + 123x^3


In [21]:
sTu = multiply_vectors(s, u, q, k, n)
d = subtract_polynomials(v, sTu, q)
print_polynomial(d)
print_polynomial(round_polynomial(d, q))

plaintext_recovered = polynomial_to_bitstring(round_polynomial(d, q), n)
print(plaintext_recovered)

4 + 60x + 79x^2 + 66x^3
x + x^2 + x^3
0111


### Example of fully functional simplified Kyber-PKE scheme

In [22]:
from random import randint
from dataclasses import dataclass

In [23]:
@dataclass(frozen=True)
class SimplifiedKyberParameters:
    q: int
    n: int
    k: int
    eta1: int
    eta2: int

In [24]:
class MathOperations:
    ''' General mathematical functions for polynomials, vectors, and matrices. '''

    @staticmethod
    def add_polynomials(f, g, modulus):
        result_length = max(len(f), len(g))
        result = [0] * result_length
        for i in range(result_length):
            coefficient_f = f[i] if i < len(f) else 0
            coefficient_g = g[i] if i < len(g) else 0
            result[i] = (coefficient_f + coefficient_g) % modulus
        return result

    @staticmethod
    def subtract_polynomials(f, g, modulus):
        result_length = max(len(f), len(g))
        result = [0] * result_length
        for i in range(result_length):
            coefficient_f = f[i] if i < len(f) else 0
            coefficient_g = g[i] if i < len(g) else 0
            result[i] = (coefficient_f - coefficient_g) % modulus
        return result
    
    @staticmethod
    def multiply_polynomials(f, g, modulus):
        result_length = len(f) + len(g) - 1
        result = [0] * result_length
        for i in range(len(f)):
            for j in range(len(g)):
                result[i+j] = (result[i+j] + f[i] * g[j]) % modulus
        return result
    
    @staticmethod
    def ring_multiply_polynomials(f, g, modulus, n):
        result = MathOperations.multiply_polynomials(f, g, modulus)
        if len(result) > n:
            for i in range(n, len(result)):
                result[i%n] = (result[i%n] - result[i]) % modulus
            result = result[:n]
        return result
    
    @staticmethod
    def add_vectors(a, b, modulus, k):
        result = []
        for i in range(k):
            result.append(MathOperations.add_polynomials(a[i], b[i], modulus))
        return result
    
    @staticmethod
    def multiply_vectors(a, b, modulus, k, n):
        result = []
        for i in range(k):
            multiplied = MathOperations.ring_multiply_polynomials(a[i], b[i], modulus, n)
            result = MathOperations.add_polynomials(result, multiplied, modulus)
        return result

    @staticmethod
    def mod_s(r, q):
        r = r % q
        if q % 2 == 0:
            if r <= q // 2:
                result = r
            else:
                result = r - q
        else:
            if r <= (q - 1) // 2:
                result = r
            else:
                result = r - q
        return result

    @staticmethod
    def round_q(x, q):
        x_prime = MathOperations.mod_s(x, q)
        if -int(q/4) <= x_prime <= int(q/4):
            return 0
        else:
            return 1
        
    @staticmethod
    def round_polynomial(f, q):
        return [MathOperations.round_q(coefficient, q) for coefficient in f]

    @staticmethod
    def multiply_polynomial_by_scalar(f, scalar, modulus):
        return [(coefficient * scalar) % modulus for coefficient in f]

    @staticmethod
    def multiply_square_matrix_by_vector(A, b, modulus, k, n):
        result = [[] for _ in range(k)]

        for row_A in range(k):
            result[row_A] = MathOperations.multiply_vectors(A[row_A], b, modulus, k, n)

        return result

    @staticmethod
    def matrix_transpose(A, k, n):
        result = [[0] * k for _ in range(n)]
        for i in range(k):
            for j in range(n):
                result[j][i] = A[i][j]
        return result
    
    @staticmethod
    def integer_inverse_mod(n, q):
        return pow(n, -1, q)

In [25]:
class Conversion:
    ''' Functions for converting between bitstrings, polynomials, and bytes representations. '''
    
    @staticmethod
    def bitstring_to_polynomial(m, n):
        f = [0] * n
        for i in range(n):
            if m[i] == "1":
                f[i] = 1
        return f

    @staticmethod
    def polynomial_to_bitstring(f, n):
        m = ""
        for i in range(n):
            if f[i] == 1:
                m += "1"
            else:
                m += "0"
        return m
    
    @staticmethod
    def text_in_bytes_to_bitstring(text_in_bytes, n):
        bitstring = ""
        
        for byte in text_in_bytes:
            binary_string = format(byte, '08b')
            bitstring += binary_string

        if len(bitstring) < n:
            bitstring += '0' * (n - len(bitstring))

        return bitstring
    
    @staticmethod
    def bitstring_to_text_in_bytes(bitstring, n):
        chunks_8bits = [bitstring[i:i+8] for i in range(0, n, 8)]
        bytes_list = bytes(int(byte, 2) for byte in chunks_8bits)
    
        return bytes_list
    
    @staticmethod
    def ints_to_bits(ints):
        bit_list = [0]*(len(ints)*8)
        for i in range(len(ints)):
            for j in range(8):
                bit_list[i*8 + j] = ints[i] % 2
                ints[i] = ints[i] // 2
        return bit_list

In [26]:
class SimplifiedCommunicationParty:
    def __init__(self, parameters: SimplifiedKyberParameters):
        self.parameters = parameters
        self.A = None
        self.s = None
        self.e = None
        self.t = None

    def generate_public_key(self):

        def select_A(q, k, n):
            A = []
            for i in range(k):
                row = []
                for j in range(k):
                    polynomial = [randint(0, q-1) for _ in range(n)]
                    row.append(polynomial)
                A.append(row)
            return A

        def select_s(k, n, eta1):
            s = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                s.append(polynomial)
            return s

        def select_e(k, n, eta1):
            e = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                e.append(polynomial)
            return e


        self.A = select_A(self.parameters.q, self.parameters.k, self.parameters.n)
        self.s = select_s(self.parameters.k, self.parameters.n, self.parameters.eta1)
        self.e = select_e(self.parameters.k, self.parameters.n, self.parameters.eta1)

        As = MathOperations.multiply_square_matrix_by_vector(self.A, self.s, self.parameters.q, self.parameters.k, self.parameters.n)
        self.t = MathOperations.add_vectors(As, self.e, self.parameters.q, self.parameters.k)

        return (self.A, self.t)

    def encrypt(self, text_in_bytes):
        def select_r(k, n, eta1):
            r = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                r.append(polynomial)
            return r

        def select_e1(k, n, eta2):
            e1 = []
            for i in range(k):
                polynomial = [randint(-eta2, eta2) for _ in range(n)]
                e1.append(polynomial)
            return e1

        def select_e2(n, eta2):
            e2 = [randint(-eta2, eta2) for _ in range(n)]
            return e2

        bitstring = Conversion.text_in_bytes_to_bitstring(text_in_bytes, self.parameters.n)
        m = Conversion.bitstring_to_polynomial(bitstring, self.parameters.n)

        r = select_r(self.parameters.k, self.parameters.n, self.parameters.eta1)
        e1 = select_e1(self.parameters.k, self.parameters.n, self.parameters.eta2)
        e2 = select_e2(self.parameters.n, self.parameters.eta2)

        AT = MathOperations.matrix_transpose(self.A, self.parameters.k, self.parameters.k)
        ATr = MathOperations.multiply_square_matrix_by_vector(AT, r, self.parameters.q, self.parameters.k, self.parameters.n)
        u = MathOperations.add_vectors(ATr, e1, self.parameters.q, self.parameters.k)

        tTr = MathOperations.multiply_vectors(self.t, r, self.parameters.q, self.parameters.k, self.parameters.n)
        half_qm = MathOperations.multiply_polynomial_by_scalar(m, (self.parameters.q+1)//2, self.parameters.q)
        tTr_e2 = MathOperations.add_polynomials(tTr, e2, self.parameters.q)
        v = MathOperations.add_polynomials(tTr_e2, half_qm, self.parameters.q)
        
        return (u, v)

    def decrypt(self, ciphertext):
        u, v = ciphertext
        sTu = MathOperations.multiply_vectors(self.s, u, self.parameters.q, self.parameters.k, self.parameters.n)
        v_sub_sTu = MathOperations.subtract_polynomials(v, sTu, self.parameters.q)
        m_recovered = MathOperations.round_polynomial(v_sub_sTu, self.parameters.q)
        bitstring_recovered = Conversion.polynomial_to_bitstring(m_recovered, self.parameters.n)
        text_in_bytes_recovered = Conversion.bitstring_to_text_in_bytes(bitstring_recovered, self.parameters.n)

        return text_in_bytes_recovered

In [27]:
q = 3329
n = 256
k = 3
eta1 = 2
eta2 = 2

In [28]:
poem_en = '''Stopping by Woods on a Snowy Evening

He gives his harness bells a shake
To ask if there is some mistake.
The only other sound's the sweep
Of easy wind and downy flake.

The woods are lovely, dark and deep,
But I have promises to keep,
And miles to go before I sleep,
And miles to go before I sleep.'''

In [29]:
poem_pl = '''Stepy akermańskie

Wpłynąłem na suchego przestwór oceanu,
Wóz nurza się w zieloność i jak łódka brodzi,
Śród fali łąk szumiących, śród kwiatów powodzi,
Omijam koralowe ostrowy burzanu.

Już mrok zapada, nigdzie drogi ni kurhanu;
Patrzę w niebo, gwiazd szukam przewodniczek łodzi;
Tam z dala błyszczy obłok? tam jutrzenka wschodzi?
To błyszczy Dniestr, to weszła lampa Akermanu'''

In [30]:
kyber_parameters = SimplifiedKyberParameters(q, n, k, eta1, eta2)
Alice = SimplifiedCommunicationParty(kyber_parameters)
Bob = SimplifiedCommunicationParty(kyber_parameters)

Alice_public_key = Alice.generate_public_key()

Bob.A = Alice_public_key[0]
Bob.t = Alice_public_key[1]

for poem in [poem_en, poem_pl]:

    Alice_recovered_bytes = b""
    poem_bytes = poem.encode('utf-8')
    for i in range(0, len(poem_bytes), kyber_parameters.n // 8):
        chunk_of_bytes = poem_bytes[i:i + kyber_parameters.n // 8]
        Bob_ciphertext = Bob.encrypt(chunk_of_bytes) 
        Alice_recovered_bytes += Alice.decrypt(Bob_ciphertext)
    
    Alice_plaintext_recovered = Alice_recovered_bytes.decode('utf-8', errors='ignore').rstrip('\x00')

    assert Alice_plaintext_recovered == poem, "Recovered plaintext is NOT the same as Original plaintext!"

    print(Alice_plaintext_recovered)
    print("\n\n")

Stopping by Woods on a Snowy Evening

He gives his harness bells a shake
To ask if there is some mistake.
The only other sound's the sweep
Of easy wind and downy flake.

The woods are lovely, dark and deep,
But I have promises to keep,
And miles to go before I sleep,
And miles to go before I sleep.



Stepy akermańskie

Wpłynąłem na suchego przestwór oceanu,
Wóz nurza się w zieloność i jak łódka brodzi,
Śród fali łąk szumiących, śród kwiatów powodzi,
Omijam koralowe ostrowy burzanu.

Już mrok zapada, nigdzie drogi ni kurhanu;
Patrzę w niebo, gwiazd szukam przewodniczek łodzi;
Tam z dala błyszczy obłok? tam jutrzenka wschodzi?
To błyszczy Dniestr, to weszła lampa Akermanu





## V2b - optimizations

In [31]:
# use 256bit ro to generate A, then public key is (ro, t) instead of (A, t)
import hashlib

In [32]:
def sampleNTT(ro,j,i, n, q):
    seed = bytes(ro+[j]+[i])
    shake = hashlib.shake_128(seed)
    j = 0
    a_hat = [0] * n
    while j < n:
        hash_bytes = shake.digest(3)
        shake.update(hash_bytes)
        hash_ints = [int(byte) for byte in hash_bytes]
        d1 = hash_ints[0] + n*(hash_ints[1]%16)
        d2 = int(hash_ints[1]//16) + 16*hash_ints[2]

        if d1 < q:
            a_hat[j] = d1
            j = j+1
        if d2 < q and j < n:
            a_hat[j] = d2
            j = j+1
    return a_hat

def generate_square_matrix_from_ro(ro, q, k, n):
    A_hat = []
    for i in range(k):
        row = []
        for j in range(k):
            polynomial = sampleNTT(ro,j,i,n,q)
            row.append(polynomial)
        A_hat.append(row)
    return A_hat

In [33]:
def print_matrix(A):
    for row in A:
        for polynomial in row:
            print_polynomial(polynomial)
        print()

In [34]:
from random import randint
ro = [randint(0, 255) for _ in range(32)]
q = 3329
n = 256
k = 3

A_hat = generate_square_matrix_from_ro(ro, q, k, n)
print(len(A_hat), len(A_hat[0]), len(A_hat[0][0]))

3 3 256


In [35]:
def compress_integer(x, q, d):
    y = (((2**d) * x) + (q//2)) // q
    return y%(2**d)

def decompress_integer(y, q, d):
    x = ((q * y) + (2**d // 2)) // (2**d)
    return x%q

In [36]:
q = 19
d = 2

for x in range(0, 19):
    y = compress_integer(x, q, d)
    x_prime = decompress_integer(y, q, d)

    print(f"x: {x}, compressed: {y}, decompressed: {x_prime}")

x: 0, compressed: 0, decompressed: 0
x: 1, compressed: 0, decompressed: 0
x: 2, compressed: 0, decompressed: 0
x: 3, compressed: 1, decompressed: 5
x: 4, compressed: 1, decompressed: 5
x: 5, compressed: 1, decompressed: 5
x: 6, compressed: 1, decompressed: 5
x: 7, compressed: 1, decompressed: 5
x: 8, compressed: 2, decompressed: 10
x: 9, compressed: 2, decompressed: 10
x: 10, compressed: 2, decompressed: 10
x: 11, compressed: 2, decompressed: 10
x: 12, compressed: 3, decompressed: 14
x: 13, compressed: 3, decompressed: 14
x: 14, compressed: 3, decompressed: 14
x: 15, compressed: 3, decompressed: 14
x: 16, compressed: 3, decompressed: 14
x: 17, compressed: 0, decompressed: 0
x: 18, compressed: 0, decompressed: 0


In [37]:
def compress_polynomial(f, q, d):
    return [compress_integer(coefficient, q, d) for coefficient in f]

def decompress_polynomial(f, q, d):
    return [decompress_integer(coefficient, q, d) for coefficient in f]

def polynomial_mod_s(f, q):
    return [mod_s(coefficient, q) for coefficient in f]

def calculate_error_polynomial(f, g, q):
    polynomial = subtract_polynomials(f, g, q)
    p_mod_s = polynomial_mod_s(polynomial, q)
    return p_mod_s

In [38]:
q = 3329
d = 10

print(f"error should be at most {(q + ((2**(d+1))//2)) // (2**(d+1))}")

f = [223,1438,3280,798]
f_compressed = compress_polynomial(f, q, d)
f_decompressed = decompress_polynomial(f_compressed, q, d)
error_polynomial = calculate_error_polynomial(f, f_decompressed, q)

print(f_compressed)
print(f_decompressed)
print(error_polynomial)

error should be at most 2
[69, 442, 1009, 245]
[224, 1437, 3280, 796]
[-1, 1, 0, 2]


In [39]:
q = 3329
d = 4

print(f"error should be at most {(q + ((2**(d+1))//2)) // (2**(d+1))}")

f = [223,1438,3280,798]
f_compressed = compress_polynomial(f, q, d)
f_decompressed = decompress_polynomial(f_compressed, q, d)
error_polynomial = calculate_error_polynomial(f, f_decompressed, q)

print(f_compressed)
print(f_decompressed)
print(error_polynomial)

error should be at most 104
[1, 7, 0, 4]
[208, 1456, 0, 832]
[15, -18, -49, -34]


In [40]:
# Ciphertext compression: (u, v) -> (c1, c2) 
# where c1 = compress(u, du) and c2 = compress(v, dv)
# ML-KEM-768 parameters: du = 10, dv = 4

In [41]:
# Fast Polynomial Multiplication -> use NTT from V4b

## V2c - full PKE scheme

- ML-KEM-768 domain parameters:
    | parameter | value |
    |-----------|-------|
    |     q     | 3329  |
    |     n     | 256   |
    |     k     | 3     |
    |     eta1  | 2     |
    |     eta2  | 2     |
    |     du    | 10    |
    |     dv    | 4     |
    ---------------------

- Kyber-PKE key generation - Alice does:
    1. select 256bit ro, compute k by k matrix A = expand(ro) 
    2. select vectors s and e - each contains k polynomials with n coefficients from -eta1 to +eta
    3. compute t = As + e
    4. encryption (public) key is (ro, t) and decryption (private) key is s

- Kyber-PKE encryption - to encrypt message (bitstring) m, Bob does:
    1. obtain a copy of Alice's public key (ro, t) and compute A = expand(ro)
    2. select vectors r and e1 - k polynomials with n coefficients (r uses eta1, e1 uses eta2) and polynomial e2 with n coefficients using eta2
    3. compute u = ATr + e1 and v = tTr + e2 + m*((q+1)//2)
    4. compute c1 = compress(u, du) and c2 = compress(v, dv)
    5. ciphertext is c = (c1, c2)

- Kyber-PKE decryption - to decrypt c=(c1, c2), Alice does:
    1. compute u' = decompress(c1, du) and v' = decompress(c2, dv)
    2. compute m = round_q(v' - sTu')

In [42]:
@dataclass(frozen=True)
class KyberParameters:
    q: int
    n: int
    k: int
    eta1: int
    eta2: int
    du: int
    dv: int
    z: int
    n_inv: int

In [43]:
from hashlib import shake_128

class NTT:
    ''' Functions for working in the Kyber Number Theoretic Transform **(NTT)** domain. '''

    @staticmethod
    def brv(v:int, n:int) -> int:
        bits_in_n = n.bit_length() - 1
        binary_string = format(v, f'0{bits_in_n}b')
        reversed_binary_string = binary_string[::-1]
        reversed_v = int(reversed_binary_string, 2)
        return reversed_v

    @staticmethod
    def generate_zetas_for_NTT(n, z, q):
        zetas = [0] * n
        
        for i in range(n):
            zetas[i] = (z**NTT.brv(i, n))%q
        return zetas

    @staticmethod
    def generate_zetas_for_multiply(n, z, q):
        zetas = [0] * n
        
        for i in range(n):
            zetas[i] = (z**(2*NTT.brv(i, n) + 1))%q
        return zetas

    @staticmethod
    def sampleNTT(ro,j,i, n, q):
        seed = bytes(ro+[j]+[i])
        shake = shake_128(seed)
        j = 0
        a_hat = [0] * n
        while j < n:
            hash_bytes = shake.digest(3)
            shake.update(hash_bytes)
            hash_ints = [int(byte) for byte in hash_bytes]
            d1 = hash_ints[0] + n*(hash_ints[1]%16)
            d2 = int(hash_ints[1]//16) + 16*hash_ints[2]

            if d1 < q:
                a_hat[j] = d1
                j = j+1
            if d2 < q and j < n:
                a_hat[j] = d2
                j = j+1
        return a_hat

    @staticmethod
    def computeNTT(f, q, n, zetas):
        ''' compute NTT of polynomial f '''
        f_hat = f
        i = 1
        length = n//2

        while length >= 2:
            start = 0
            while start < n:
                zeta = zetas[i]
                i = i+1
                for j in range(start, start+length):
                    t = (zeta * f_hat[j+length]) % q
                    f_hat[j+length] = (f_hat[j] - t) % q
                    f_hat[j] = (f_hat[j] + t) % q
                start = start + 2*length
            length = length // 2
        
        return f_hat

    @staticmethod
    def computeNTT_inverse(f_hat, q, n, n_inv, zetas):
        f = f_hat
        i = n//2 - 1
        length = 2
        while length <= n//2:
            start = 0
            while start < n:
                zeta = zetas[i]
                i = i-1
                for j in range(start, start+length):
                    t = f[j]
                    f[j] = (t + f[j+length]) % q
                    f[j+length] = (zeta * (f[j+length] - t)) % q
                start = start + 2*length
            length = 2*length

        f = [(coefficient * n_inv) % q for coefficient in f]

        return f

    @staticmethod
    def compute_vector_NTT(a, q, k, n, zetas):
        ''' compute NTT of k-length vector a '''
        a_hat = [0]*k
        for i in range(k):
            polynomial_hat = NTT.computeNTT(a[i], q, n, zetas)
            a_hat[i] = polynomial_hat
        return a_hat

    @staticmethod
    def compute_vector_NTT_inverse(a_hat, q, k, n, n_inv, zetas):
        ''' compute inverse NTT of k-length vector a_hat '''
        a = [0]*k
        for i in range(k):
            polynomial = NTT.computeNTT_inverse(a_hat[i], q, n, n_inv, zetas)
            a[i] = polynomial
        return a

    @staticmethod
    def base_case_multiply(a0, a1, b0, b1, q, z):
        c0 = (a0*b0 + a1*b1*z) % q
        c1 = (a0*b1 + a1*b0) % q
        return c0, c1

    @staticmethod
    def multiplyNTT(f_hat, g_hat, q, n, zetas):
        ''' multiply two polynomials in NTT domain '''
        h = [0] * n
        for i in range(n//2):
            a0 = f_hat[2*i]
            a1 = f_hat[2*i + 1]
            b0 = g_hat[2*i]
            b1 = g_hat[2*i + 1]
            h[2*i], h[2*i + 1] = NTT.base_case_multiply(a0, a1, b0, b1, q, zetas[i])

        return h

    @staticmethod
    def multiply_vectors_NTT(a_hat, b_hat, q, k, n, zetas):
        ''' multiply two k-length vectors in NTT domain '''
        result = []
        for i in range(k):
            multiplied = NTT.multiplyNTT(a_hat[i], b_hat[i], q, n, zetas)
            result = MathOperations.add_polynomials(result, multiplied, q)
        return result

    @staticmethod
    def multiply_square_matrix_by_vector_NTT(A_hat, b_hat, q, k, n, zetas):
        ''' multiply k x k matrix by k-length vector in NTT domain '''
        result = [[] for _ in range(k)]

        for row_A in range(len(A_hat)):
            result[row_A] = NTT.multiply_vectors_NTT(A_hat[row_A], b_hat, q, k, n, zetas)

        return result

    @staticmethod
    def generate_square_matrix_NTT_from_ro(ro, q, k, n):
        A_hat = []
        for i in range(k):
            row = []
            for j in range(k):
                polynomial = NTT.sampleNTT(ro,j,i,n,q)
                row.append(polynomial)
            A_hat.append(row)
        return A_hat

In [44]:
class Compression:
    ''' Functions used for compression / decompression '''

    @staticmethod
    def compress_integer(x, q, d):
        y = (((2**d) * x) + (q//2)) // q
        return y%(2**d)

    @staticmethod
    def decompress_integer(y, q, d):
        x = ((q * y) + (2**d // 2)) // (2**d)
        return x%q

    @staticmethod
    def compress_polynomial(f, q, d):
        return [Compression.compress_integer(coefficient, q, d) for coefficient in f]

    @staticmethod
    def decompress_polynomial(f, q, d):
        return [Compression.decompress_integer(coefficient, q, d) for coefficient in f]

    @staticmethod
    def compress_vector(a, q, d):
        return [Compression.compress_polynomial(polynomial, q, d) for polynomial in a]

    @staticmethod
    def decompress_vector(a, q, d):
        return [Compression.decompress_polynomial(polynomial, q, d) for polynomial in a]

In [45]:
class CommunicationParty:
    def __init__(self, parameters: KyberParameters):
        self.parameters = parameters
        self.A_hat = None
        self.s_hat = None
        self.e_hat = None
        self.t_hat = None
        self.ro = None
        self.zetas_for_NTT = NTT.generate_zetas_for_NTT(self.parameters.n//2, self.parameters.z, self.parameters.q)
        self.zetas_for_multiply = NTT.generate_zetas_for_multiply(self.parameters.n//2, self.parameters.z, self.parameters.q)

    def generate_public_key(self):

        def select_ro(n):
            ro_length = n // (n.bit_length()-1)
            return [randint(0, n-1) for _ in range(ro_length)]

        def select_s(k, n, eta1):
            s = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                s.append(polynomial)
            return s

        def select_e(k, n, eta1):
            e = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                e.append(polynomial)
            return e

        self.ro = select_ro(self.parameters.n)
        self.A_hat = NTT.generate_square_matrix_NTT_from_ro(self.ro, self.parameters.q, self.parameters.k, self.parameters.n)

        s = select_s(self.parameters.k, self.parameters.n, self.parameters.eta1)
        e = select_e(self.parameters.k, self.parameters.n, self.parameters.eta1)
        
        self.s_hat = NTT.compute_vector_NTT(s, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_NTT)
        self.e_hat = NTT.compute_vector_NTT(e, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_NTT)

        A_hat_s_hat = NTT.multiply_square_matrix_by_vector_NTT(self.A_hat, self.s_hat, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_multiply)

        self.t_hat = MathOperations.add_vectors(A_hat_s_hat, self.e_hat, self.parameters.q, self.parameters.k)

        return (self.ro, self.t_hat)

    def obtain_key(self, key):
        self.ro = key[0]
        self.t_hat = key[1]
        self.A_hat = NTT.generate_square_matrix_NTT_from_ro(self.ro, self.parameters.q, self.parameters.k, self.parameters.n)

    def encrypt(self, text_in_bytes):
        def select_r(k, n, eta1):
            r = []
            for i in range(k):
                polynomial = [randint(-eta1, eta1) for _ in range(n)]
                r.append(polynomial)
            return r

        def select_e1(k, n, eta2):
            e1 = []
            for i in range(k):
                polynomial = [randint(-eta2, eta2) for _ in range(n)]
                e1.append(polynomial)
            return e1

        def select_e2(n, eta2):
            e2 = [randint(-eta2, eta2) for _ in range(n)]
            return e2

        bitstring = Conversion.text_in_bytes_to_bitstring(text_in_bytes, self.parameters.n)
        m  = Conversion.bitstring_to_polynomial(bitstring, self.parameters.n)

        r = select_r(self.parameters.k, self.parameters.n, self.parameters.eta1)
        e1 = select_e1(self.parameters.k, self.parameters.n, self.parameters.eta2)
        e2 = select_e2(self.parameters.n, self.parameters.eta2)

        r_hat = NTT.compute_vector_NTT(r, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_NTT)

        A_hatT = MathOperations.matrix_transpose(self.A_hat, self.parameters.k, self.parameters.k)
        A_hatT_r = NTT.multiply_square_matrix_by_vector_NTT(A_hatT, r_hat, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_multiply)
        ntt_inv_A_hatT_r = NTT.compute_vector_NTT_inverse(A_hatT_r, self.parameters.q, self.parameters.k, self.parameters.n, self.parameters.n_inv, self.zetas_for_NTT)
        u = MathOperations.add_vectors(ntt_inv_A_hatT_r, e1, self.parameters.q, self.parameters.k)

        t_hatT_r_hat = NTT.multiply_vectors_NTT(self.t_hat, r_hat, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_multiply)
        ntt_inv_t_hatT_r_hat = NTT.computeNTT_inverse(t_hatT_r_hat, self.parameters.q, self.parameters.n, self.parameters.n_inv, self.zetas_for_NTT)
        half_qm = MathOperations.multiply_polynomial_by_scalar(m, (self.parameters.q+1)//2, self.parameters.q)
        ntt_inv_t_hatT_r_hat_e2 = MathOperations.add_polynomials(ntt_inv_t_hatT_r_hat, e2, self.parameters.q)
        v = MathOperations.add_polynomials(ntt_inv_t_hatT_r_hat_e2, half_qm, self.parameters.q)
        
        c1 = Compression.compress_vector(u, self.parameters.q, self.parameters.du)
        c2 = Compression.compress_polynomial(v, self.parameters.q, self.parameters.dv)

        return (c1, c2)

    def decrypt(self, ciphertext):
        c1, c2 = ciphertext
        u_prim = Compression.decompress_vector(c1, self.parameters.q, self.parameters.du)
        v_prim = Compression.decompress_polynomial(c2, self.parameters.q, self.parameters.dv)
        ntt_u_prim = NTT.compute_vector_NTT(u_prim, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_NTT)
        s_hatT_ntt_u_prim = NTT.multiply_vectors_NTT(self.s_hat, ntt_u_prim, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_multiply)
        ntt_inv_s_hatT_ntt_u_prim = NTT.computeNTT_inverse(s_hatT_ntt_u_prim, self.parameters.q, self.parameters.n, self.parameters.n_inv, self.zetas_for_NTT)
        w = MathOperations.subtract_polynomials(v_prim, ntt_inv_s_hatT_ntt_u_prim, self.parameters.q)
        
        m_recovered = MathOperations.round_polynomial(w, self.parameters.q)
        bitstring_recovered = Conversion.polynomial_to_bitstring(m_recovered, self.parameters.n)
        text_in_bytes_recovered = Conversion.bitstring_to_text_in_bytes(bitstring_recovered, self.parameters.n)

        return text_in_bytes_recovered

In [46]:
q = 3329
n = 256
k = 3
eta1 = 2
eta2 = 2
du = 10
dv = 4
z = 17
n_inv = MathOperations.integer_inverse_mod(n//2, q)

In [47]:
poem_en = '''Stopping by Woods on a Snowy Evening

He gives his harness bells a shake
To ask if there is some mistake.
The only other sound's the sweep
Of easy wind and downy flake.

The woods are lovely, dark and deep,
But I have promises to keep,
And miles to go before I sleep,
And miles to go before I sleep.'''

poem_pl = '''Stepy akermańskie

Wpłynąłem na suchego przestwór oceanu,
Wóz nurza się w zieloność i jak łódka brodzi,
Śród fali łąk szumiących, śród kwiatów powodzi,
Omijam koralowe ostrowy burzanu.

Już mrok zapada, nigdzie drogi ni kurhanu;
Patrzę w niebo, gwiazd szukam przewodniczek łodzi;
Tam z dala błyszczy obłok? tam jutrzenka wschodzi?
To błyszczy Dniestr, to weszła lampa Akermanu'''

In [48]:
kyber_parameters = KyberParameters(q, n, k, eta1, eta2, du, dv, z, n_inv)
Alice = CommunicationParty(kyber_parameters)
Bob = CommunicationParty(kyber_parameters)

Alice_public_key = Alice.generate_public_key()

Bob.obtain_key(Alice_public_key)

for poem in [poem_en, poem_pl]:

    Alice_recovered_bytes = b""
    poem_bytes = poem.encode('utf-8')
    for i in range(0, len(poem_bytes), kyber_parameters.n // 8):
        chunk_of_bytes = poem_bytes[i:i + kyber_parameters.n // 8]
        Bob_ciphertext = Bob.encrypt(chunk_of_bytes) 
        Alice_recovered_bytes += Alice.decrypt(Bob_ciphertext)
    
    Alice_plaintext_recovered = Alice_recovered_bytes.decode('utf-8', errors='ignore').rstrip('\x00')

    assert Alice_plaintext_recovered == poem, "Recovered plaintext is NOT the same as Original plaintext!"

    print(Alice_plaintext_recovered)
    print("\n\n")

Stopping by Woods on a Snowy Evening

He gives his harness bells a shake
To ask if there is some mistake.
The only other sound's the sweep
Of easy wind and downy flake.

The woods are lovely, dark and deep,
But I have promises to keep,
And miles to go before I sleep,
And miles to go before I sleep.



Stepy akermańskie

Wpłynąłem na suchego przestwór oceanu,
Wóz nurza się w zieloność i jak łódka brodzi,
Śród fali łąk szumiących, śród kwiatów powodzi,
Omijam koralowe ostrowy burzanu.

Już mrok zapada, nigdzie drogi ni kurhanu;
Patrzę w niebo, gwiazd szukam przewodniczek łodzi;
Tam z dala błyszczy obłok? tam jutrzenka wschodzi?
To błyszczy Dniestr, to weszła lampa Akermanu





## V2d - Kyber KEM

- ML-KEM-768 domain parameters:

    | parameter | value |
    |-----------|-------|
    |     q     | 3329  |
    |     n     | 256   |
    |     k     | 3     |
    |     eta1  | 2     |
    |     eta2  | 2     |
    |     du    | 10    |
    |     dv    | 4     |
-----------------------------

- Fujisaki-Okamoto Transform:
    - H(s) = SHA3-256(s)
    - J(s) = SHAKE256(s, 8*32)
    - G(c) = SHA3-512(c)


- Kyber-KEM key generation - Alice does:
    1. use Kyber-PKE key generation to select Kyber-PKE encryption key ek=(ro, t) and decryption key s
    2. select 256bit z
    3. encapsulation key is ek = (ro, t) and decapsulation key is dk = (s, ek, H(ek), z)


- Kyber-KEM encapsulation - to establish a shared secret key with Alice, Bob does:
    1. obtain a copy of Alice's encapsulation key ek
    2. select 256bit m
    3. compute h = H(ek) and (K, R) = G(m,h) where K,R are 256bit
    4. use Kyber-PKE encryption to encrypt m with key ek, using R to generate random quantities needed, then call the resulting ciphertext c
    5. output the shared secret key K and ciphertext c 


- Kyber-KEM decapsulation - to recover secret key K from c using dk=(s,ek,H(ek),z), Alice does:
    1. use Kyber-PKE decryption to decrypt c with key s, call the resulting plaintext m'
    2. compute (K', R') = G(m', H(ek))
    3. compute K_bar = J(z, c)
    4. use Kyber-PKE encryption to encrypt m' with key ek, using R' to generate random quantities needed, then call the resulting ciphertext c'
    5. if c is NOT equal c' then return K_bar
    6. if c is equal c' then return K'

In [49]:
from hashlib import sha3_256, sha3_512, shake_128, shake_256 

class Hash:
    ''' Hash functions used in Kyber '''

    @staticmethod
    def H(s):
        sha = sha3_256(s)
        hash_bytes = sha.digest()
        hash_ints = [int(byte) for byte in hash_bytes]
        return hash_ints
        
    @staticmethod
    def J(s):
        shake = shake_256(s)
        hash_bytes = shake.digest(32)
        hash_ints = [int(byte) for byte in hash_bytes]
        return hash_ints

    @staticmethod
    def G(c):
        sha = sha3_512(c)
        hash_bytes = sha.digest()
        hash_ints = [int(byte) for byte in hash_bytes]
        a = hash_ints[:32]
        b = hash_ints[32:]
        return (a, b)

In [50]:
class Randomness:
    ''' Random and Pseudo-random functions used in Kyber '''

    @staticmethod
    def PRF(eta, s, b):
        seed = bytes(s + [b])
        shake = shake_256(seed)
        hash_bytes = shake.digest(64*eta)
        hash_ints = [int(byte) for byte in hash_bytes]
        return hash_ints
    
    @staticmethod
    def samplePolyCBD(q, n, eta, ints):
        b = Conversion.ints_to_bits(ints)

        f = [0] * n
        for i in range(n):
            x = 0
            y = 0
            for j in range(eta):
                x += b[2*i*eta + j]
                y += b[2*i*eta + eta + j]
            f[i] = (x - y) % q
        return f
    
    @staticmethod
    def generate_vector(q, k, n, eta, randomness, n_counter):
        y = []
        for i in range(k):
            polynomial = Randomness.generate_polynomial(q, n, eta, randomness, n_counter)
            y.append(polynomial)
            n_counter += 1
        return y

    @staticmethod
    def generate_polynomial(q, n, eta, randomness, n_counter):
        ints = Randomness.PRF(eta, randomness, n_counter)
        polynomial = Randomness.samplePolyCBD(q, n, eta, ints)
        return polynomial
    
    @staticmethod
    def random_bytes(n):
        num_of_bytes = n // (n.bit_length()-1)
        ints = [randint(0, n-1) for _ in range(num_of_bytes)]
        return bytes(ints)
    
    @staticmethod
    def random_ints(n):
        num_of_bytes = n // (n.bit_length()-1)
        ints = [randint(0, n-1) for _ in range(num_of_bytes)]
        return ints


In [51]:
class CommunicationPartyKEM:
    def __init__(self, parameters: KyberParameters):
        self.dk = None

        self.ek = None
        self.A_hat = None
        self.parameters = parameters
        self.zetas_for_NTT = NTT.generate_zetas_for_NTT(self.parameters.n//2, self.parameters.z, self.parameters.q)
        self.zetas_for_multiply = NTT.generate_zetas_for_multiply(self.parameters.n//2, self.parameters.z, self.parameters.q)

    def key_generation(self):

        d_bytes = Randomness.random_bytes(self.parameters.n)
        z_bytes = Randomness.random_bytes(self.parameters.n)
        ek_PKE, dk_PKE = self.key_generation_internal(d_bytes)

        ek_ints = ek_PKE[0] + [coefficient for polynomial in ek_PKE[1] for coefficient in polynomial]
        ek_bytes = b''.join(num.to_bytes(2, 'big') for num in ek_ints)
        H_ek = Hash.H(ek_bytes)
        dk = (dk_PKE, ek_PKE, H_ek, z_bytes)

        self.dk = dk
        self.ek = ek_PKE

        return ek_PKE

    def key_generation_internal(self, d_bytes):

        ro, sigma = Hash.G(d_bytes + bytes([self.parameters.k]))

        self.A_hat = NTT.generate_square_matrix_NTT_from_ro(ro, self.parameters.q, self.parameters.k, self.parameters.n)

        n_counter = 0

        s = Randomness.generate_vector(self.parameters.q, self.parameters.k, self.parameters.n, self.parameters.eta1, sigma, n_counter)
        n_counter += self.parameters.k
        e = Randomness.generate_vector(self.parameters.q, self.parameters.k, self.parameters.n, self.parameters.eta1, sigma, n_counter)

        s_hat = NTT.compute_vector_NTT(s, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_NTT)
        e_hat = NTT.compute_vector_NTT(e, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_NTT)

        A_hat_s_hat = NTT.multiply_square_matrix_by_vector_NTT(self.A_hat, s_hat, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_multiply)

        t_hat = MathOperations.add_vectors(A_hat_s_hat, e_hat, self.parameters.q, self.parameters.k)

        ek = (ro, t_hat)
        dk = s_hat

        return (ek, dk)

    def obtain_key(self, key):
        self.ek = key
        ro = key[0]
        self.A_hat = NTT.generate_square_matrix_NTT_from_ro(ro, self.parameters.q, self.parameters.k, self.parameters.n)

    def encrypt(self, text_in_bytes, randomness):

        bitstring = Conversion.text_in_bytes_to_bitstring(text_in_bytes, self.parameters.n)
        m  = Conversion.bitstring_to_polynomial(bitstring, self.parameters.n)

        n_counter = 0

        r = Randomness.generate_vector(self.parameters.q, self.parameters.k, self.parameters.n, self.parameters.eta1, randomness, n_counter)
        n_counter += self.parameters.k

        e1 = Randomness.generate_vector(self.parameters.q, self.parameters.k, self.parameters.n, self.parameters.eta2, randomness, n_counter)
        n_counter += self.parameters.k

        e2 = Randomness.generate_polynomial(self.parameters.q, self.parameters.n, self.parameters.eta2, randomness, n_counter)
        
        r_hat = NTT.compute_vector_NTT(r, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_NTT)

        A_hatT = MathOperations.matrix_transpose(self.A_hat, self.parameters.k, self.parameters.k)
        A_hatT_r = NTT.multiply_square_matrix_by_vector_NTT(A_hatT, r_hat, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_multiply)
        ntt_inv_A_hatT_r = NTT.compute_vector_NTT_inverse(A_hatT_r, self.parameters.q, self.parameters.k, self.parameters.n, self.parameters.n_inv, self.zetas_for_NTT)
        u = MathOperations.add_vectors(ntt_inv_A_hatT_r, e1, self.parameters.q, self.parameters.k)

        t_hat = self.ek[1]
        t_hatT_r_hat = NTT.multiply_vectors_NTT(t_hat, r_hat, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_multiply)
        ntt_inv_t_hatT_r_hat = NTT.computeNTT_inverse(t_hatT_r_hat, self.parameters.q, self.parameters.n, self.parameters.n_inv, self.zetas_for_NTT)
        half_qm = MathOperations.multiply_polynomial_by_scalar(m, (self.parameters.q+1)//2, self.parameters.q)
        ntt_inv_t_hatT_r_hat_e2 = MathOperations.add_polynomials(ntt_inv_t_hatT_r_hat, e2, self.parameters.q)
        v = MathOperations.add_polynomials(ntt_inv_t_hatT_r_hat_e2, half_qm, self.parameters.q)
        
        c1 = Compression.compress_vector(u, self.parameters.q, self.parameters.du)
        c2 = Compression.compress_polynomial(v, self.parameters.q, self.parameters.dv)

        return (c1, c2)
    
    def encapsulate(self):

        m_bytes = Randomness.random_bytes(self.parameters.n)

        ek_ints = self.ek[0] + [coefficient for polynomial in self.ek[1] for coefficient in polynomial]
        ek_bytes = b''.join(num.to_bytes(2, 'big') for num in ek_ints)
        H_ek = Hash.H(ek_bytes)

        H_ek_bytes = bytes(H_ek)

        K, r = Hash.G(m_bytes + H_ek_bytes)

        c = self.encrypt(m_bytes, r)
        
        return (K, c)

    def decrypt(self, ciphertext):
        c1, c2 = ciphertext
        u_prim = Compression.decompress_vector(c1, self.parameters.q, self.parameters.du)
        v_prim = Compression.decompress_polynomial(c2, self.parameters.q, self.parameters.dv)
        ntt_u_prim = NTT.compute_vector_NTT(u_prim, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_NTT)
        s_hat = self.dk[0]
        s_hatT_ntt_u_prim = NTT.multiply_vectors_NTT(s_hat, ntt_u_prim, self.parameters.q, self.parameters.k, self.parameters.n, self.zetas_for_multiply)
        ntt_inv_s_hatT_ntt_u_prim = NTT.computeNTT_inverse(s_hatT_ntt_u_prim, self.parameters.q, self.parameters.n, self.parameters.n_inv, self.zetas_for_NTT)
        w = MathOperations.subtract_polynomials(v_prim, ntt_inv_s_hatT_ntt_u_prim, self.parameters.q)
        
        m_recovered = MathOperations.round_polynomial(w, self.parameters.q)
        bitstring_recovered = Conversion.polynomial_to_bitstring(m_recovered, self.parameters.n)
        text_in_bytes_recovered = Conversion.bitstring_to_text_in_bytes(bitstring_recovered, self.parameters.n)

        return text_in_bytes_recovered
    
    def decapsulate(self, ciphertext):
        m_prime = self.decrypt(ciphertext)

        ek_ints = self.ek[0] + [coefficient for polynomial in self.ek[1] for coefficient in polynomial]
        ek_bytes = b''.join(num.to_bytes(2, 'big') for num in ek_ints)
        H_ek = Hash.H(ek_bytes)
        
        H_ek_bytes = bytes(H_ek)

        K_prime, r_prime = Hash.G(m_prime + H_ek_bytes)

        c_ints = [coefficient for polynomial in ciphertext[0] for coefficient in polynomial] + ciphertext[1]
        c_bytes = b''.join(num.to_bytes(2, 'big') for num in c_ints)
        z_bytes = self.dk[3]
        K_bar = Hash.J(z_bytes + c_bytes)

        c_prime = self.encrypt(m_prime, r_prime)

        if ciphertext != c_prime:
            return K_bar
        else:
            return K_prime


In [52]:
kyber_parameters = KyberParameters(q, n, k, eta1, eta2, du, dv, z, n_inv)
Alice = CommunicationPartyKEM(kyber_parameters)
Bob = CommunicationPartyKEM(kyber_parameters)

Alice_public_key = Alice.key_generation()

Bob.obtain_key(Alice_public_key)

Bob_shared_secret, Bob_ciphertext = Bob.encapsulate()
Alice_shared_secret = Alice.decapsulate(Bob_ciphertext)

assert Alice_shared_secret == Bob_shared_secret, "Shared secrets do not match!"

if Alice_shared_secret == Bob_shared_secret:
    print("Shared secrets match!")

Shared secrets match!


# V3

## V3a - Dilithium (Toy version)

In [53]:
q = 2**23 - 2**13 + 1
n = 256
k = 8
l = 7
eta = 2
gamma1 = 2**19
tau = 60
beta = 120

In [54]:
def select_A(q, k, l, n):
    A = []
    for i in range(k):
        row = []
        for j in range(l):
            polynomial = [randint(0, q-1) for _ in range(n)]
            row.append(polynomial)
        A.append(row)
    return A

def select_s1(l, n, eta):
    s = []
    for i in range(l):
        polynomial = [randint(-eta, eta) for _ in range(n)]
        s.append(polynomial)
    return s

def select_s2(k, n, eta):
    s = []
    for i in range(k):
        polynomial = [randint(-eta, eta) for _ in range(n)]
        s.append(polynomial)
    return s

In [55]:
def multiply_matrix_by_vector(A, b, modulus, k, l, n):
        result = [[] for _ in range(k)]

        for row_A in range(k):
            result[row_A] = MathOperations.multiply_vectors(A[row_A], b, modulus, l, n)

        return result

In [56]:
A = select_A(q, k, l, n)
s1 = select_s1(l, n, eta)
s2 = select_s2(k, n, eta)

# A (k x l) * s1 (l x 1) = As1 (k x 1)
# As1 (k x 1) + s2 (k x 1) = t (k x 1)

As1 = multiply_matrix_by_vector(A, s1, q, k, l, n)
t = MathOperations.add_vectors(As1, s2, q, k)

# public key = (A, t)
# private key = (s1, s2)

In [57]:
def select_y(l, n, gamma1):
    y = []
    for i in range(l):
        polynomial = [randint(-gamma1+1, gamma1) for _ in range(n)]
        y.append(polynomial)
    return y

In [58]:
def multiply_vector_by_polynomial(a, f, modulus, l, n):
    result = [[0]*n] * l
    for i in range(l):
        result[i] = MathOperations.ring_multiply_polynomials(a[i], f, modulus, n)
        
    return result

def H():
    pass

def HighBits(w):
    pass

def LowBits(w):
    pass

In [59]:
# M = "1010101"

# y = select_y(l, n, gamma1) # (l x 1)

# w = multiply_matrix_by_vector(A, y, q, k, l, n) # (k x 1)

# w1 = HighBits(w) # (k x 1)

# c = H(M, w1) # (1 x 1)

# cs1 = multiply_vector_by_polynomial(c, s1, q, l, n) # (l x 1)

# z = add_vectors(cs1, y, q, l) # (l x 1)

# if LowBits(w - cs2) are "small enough":
#   signature sigma = (c, z)

In [60]:
# Az = multiply_matrix_by_vector(A, z, q, k, l, n) # (k x 1)
# ct = multiply_vector_by_polynomial(c, t, q, k, n) # (k x 1)
# Az_sub_ct = subtract_vectors(Az, ct, q, k)

# w1_prime = HighBits(Az_sub_ct) # (k x 1)

# if c == H(M, w1_prime):
#   signature is valid

## V3b - Dilithium scheme (without t compression)

- ML-DSA-87 domain parameters:

    | parameter | value   |
    |-----------|---------|
    |   n       | 256     |
    |   q       | 8380417 |
    |   z       | 1753    |
    |   d       | 13      |
    |   tau     | 60      |
    |   lamb    | 256     |
    |   gamma1  | 2**19   |
    |   gamma2  | (q-1)/32|
    |   k       | 8       |
    |   l       | 7       |
    |   eta     | 2       |
    |   beta    | 120     |
    |   omega   | 75      |
    |   alpha   | 2*gamma2|
-----------------------------

- Dilithium key generation - Alice does:
    1. select random seed 256bit Ksi
    2. compute (256bit ro, 512bit ro', 256bit K) = H(Ksi, 1024)
    3. compute k by l matrix A = ExpandA(ro)
    4. compute (l length s1, k lengths s2) = ExpandS(ro'), vectors of coefficients in [-eta, eta]
    5. compute t = As1 + s2
    6. compute tr = H(ro||t, 2*lamb)
    7. verification key is PK = (ro, t) and signature key is SK = (ro, K, tr, s1, s2)

- Dilithium signature generation - to sign message (bitstring) M, XXX does:
    1. compute k by l matrix A = ExpandA(ro)
    2. compute 512bit message representative miu = H(tr||M, 512)
    3. compute 512bit ro'' = H(K||rnd||miu, 512) with rnd is 256bit 0-string or random
    4. set kappa = 0
    5. set Found = false
    6. While Fould = false do:
        
        a. compute l length vector of polynomials y = ExpandMask(ro'', kappa) with coefficients in (-gamma1, gamma1]
        
        b. compute w = Ay and w1 = HighBits(w, 2*gamma2)
        
        c. compute c_tilda = H(miu||w1, 2*lamb)
        
        d. compute c = SampleInBall(c_tilda) and z = y + cs1
        
        e. compute r0 = LowBits(w - cs2, 2*gamma2)
        
        f. if size(z) < gamma1 - beta AND size(r0) < gamma2 - beta then Found = true
        
        g. kappa = kappa + l
        
    7. signature sigma = (c_tilda, z)

- Dilithium signature verification - to verify XXX's signature sigma = (c_tilda, z) on M, YYY does:
    1. obtain a copy of public key PK = (ro, t)
    2. check that size(z) < gamma1 - beta (if not then reject)
    3. compute k by l matrix A = ExpandA(ro)
    4. compute compute tr = H(ro||t, 2*lamb) and miu = H(tr||M, 512)
    5. compute c = SampleInBall(c_tilda)
    6. compute w1' = HighBits(Az - ct, 2*gamma2)
    7. check that c_tilda = H(miu||w1', 2*lamb) (if not then reject)
    8. accept the signature


In [61]:
def decompose(r, alpha):
    r0 = mod_s(r, alpha)
    r1 = (r - r0) // alpha
    return r0, r1

def LowBits(r, alpha):
    return decompose(r, alpha)[0]

def HighBits(r, alpha):
    return decompose(r, alpha)[1]

def polynomialHighBits(f, alpha):
    return [HighBits(coefficient, alpha) for coefficient in f]

def vectorHighBits(w, alpha):
    return [polynomialHighBits(polynomial, alpha) for polynomial in w]

def polynomialLowBits(f, alpha):
    return [LowBits(coefficient, alpha) for coefficient in f]

def vectorLowBits(w, alpha):
    return [polynomialLowBits(polynomial, alpha) for polynomial in w]
    

In [62]:
q = 21
alpha = 4

for r in range(0, q):
    r0, r1 = decompose(r, alpha)
    print(f"r: {r}, r1: {r1}, r0: {r0}")

r: 0, r1: 0, r0: 0
r: 1, r1: 0, r0: 1
r: 2, r1: 0, r0: 2
r: 3, r1: 1, r0: -1
r: 4, r1: 1, r0: 0
r: 5, r1: 1, r0: 1
r: 6, r1: 1, r0: 2
r: 7, r1: 2, r0: -1
r: 8, r1: 2, r0: 0
r: 9, r1: 2, r0: 1
r: 10, r1: 2, r0: 2
r: 11, r1: 3, r0: -1
r: 12, r1: 3, r0: 0
r: 13, r1: 3, r0: 1
r: 14, r1: 3, r0: 2
r: 15, r1: 4, r0: -1
r: 16, r1: 4, r0: 0
r: 17, r1: 4, r0: 1
r: 18, r1: 4, r0: 2
r: 19, r1: 5, r0: -1
r: 20, r1: 5, r0: 0


In [63]:
# import hashlib

# def SHAKE256(M, d):
#     shake = hashlib.shake_256(M)
#     length_in_bytes = d // 8
#     return shake.digest(length_in_bytes)

# def H(M, d):
#     hash_in_bytes = SHAKE256(M, d)
#     hash_in_bits = ''.join(format(byte, '08b') for byte in hash_in_bytes)

#     assert len(hash_in_bits) == d, f"Expected hash length: {d}, but got: {len(hash_in_bits)}"

#     return hash_in_bits

In [64]:
q = 2**23 - 2**13 + 1
n = 256
k = 8
l = 7
eta = 2
gamma1 = 2**19
tau = 60
beta = 120
gamma2 = (q-1)//32
lamb = 256

### simplified example

In [65]:
q = 16417
n = 4
k = 3
l = 2
eta = 10
gamma1 = 1024
tau = 4
beta = 40
gamma2 = (q-1)//32
alpha = 2*gamma2
m = 16

In [66]:
A = [[[15196, 7926, 8057, 13612], [14303, 5, 12257, 2347]],
     [[9765, 10436, 10983, 5860], [5393, 311, 5144, 10841]],
     [[11868, 7995, 10716, 3121], [9390, 2055, 6505, 2440]]]

s1 = [[2,5,-10,-5], [2,3,-4,-4]]

s2 = [[-8,3,4,4], [8,10,1,8], [-3,-8,6,6]]

As1 = multiply_matrix_by_vector(A, s1, q, k, l, n)

t = MathOperations.add_vectors(As1, s2, q, k)

print_vector(t)


5384 + 8401x + 14221x^2 + 11425x^3
4578 + 1291x + 5976x^2 + 9841x^3
3959 + 14751x + 15381x^2 + 14072x^3



In [67]:
y = [[707,-155,357,-822], [474,-566,-869,-542]]

w = multiply_matrix_by_vector(A, y, q, k, l, n)

print_vector(w)

w1 = vectorHighBits(w, 2*gamma2)

print_vector(w1)


7023 + 3184x + 4074x^2 + 5566x^3
9495 + 7254x + 7431x^2 + 13483x^3
4189 + 7420x + 13635x^2 + 4161x^3


7 + 3x + 4x^2 + 5x^3
9 + 7x + 7x^2 + 13x^3
4 + 7x + 13x^2 + 4x^3



In [68]:
def vector_mod_s(w, q):
    return [polynomial_mod_s(polynomial, q) for polynomial in w]

In [69]:
# c_tilda = H(miu || w1)
# c = SampleInBall(c_tilda)
# suppose this is the c:
c = [-1,-1,1,-1]

cs1 = multiply_vector_by_polynomial(s1, c, q, l, n)

z = MathOperations.add_vectors(cs1, y, q, l)

print_vector(z)

z_mod_s = vector_mod_s(z, q)

print_vector(z_mod_s)

print(f"z vector size: {vector_size(z_mod_s, q)} should be less than {gamma1 - beta}")


715 + 16250x + 359x^2 + 15613x^3
475 + 15846x + 15547x^2 + 15884x^3


715 + -167x + 359x^2 + -804x^3
475 + -571x + -870x^2 + -533x^3

z vector size: 870 should be less than 984


In [70]:
cs2 = multiply_vector_by_polynomial(s2, c, q, k, n)

w_sub_cs2 = subtract_vectors(w, cs2, q, k)

print_vector(w_sub_cs2)

r0 = vectorLowBits(w_sub_cs2, 2*gamma2)

print_vector(r0)

print(f"r0 vector size: {vector_size(r0, q)} should be less than {gamma2 - beta}")


7012 + 3179x + 4085x^2 + 5563x^3
9486 + 7279x + 7426x^2 + 13490x^3
4194 + 7409x + 13630x^2 + 4178x^3


-170 + 101x + -19x^2 + 433x^3
252 + 97x + 244x^2 + 152x^3
90 + 227x + 292x^2 + 74x^3

r0 vector size: 433 should be less than 473


In [71]:
# c = SampleInBall(c_tilda)
# suppose this is the c:
c = [-1,-1,1,-1]

Az = multiply_matrix_by_vector(A, z, q, k, l, n)

ct = multiply_vector_by_polynomial(t, c, q, k, n)

Az_sub_ct = subtract_vectors(Az, ct, q, k)

print_vector(Az_sub_ct)

w1_prime = vectorHighBits(Az_sub_ct, 2*gamma2)

print_vector(w1_prime)

if w1_prime == w1:
    print("signature accepted")


7012 + 3179x + 4085x^2 + 5563x^3
9486 + 7279x + 7426x^2 + 13490x^3
4194 + 7409x + 13630x^2 + 4178x^3


7 + 3x + 4x^2 + 5x^3
9 + 7x + 7x^2 + 13x^3
4 + 7x + 13x^2 + 4x^3

signature accepted


## V3c - Dilithium t compression

In [72]:
def power2round(r, d):
    r0 = mod_s(r, 2**d)
    r1 = ((r - r0)+(2**d)//2) // (2**d)
    return r0, r1

In [73]:
q = 23
d = 3

for i in range(0, q):
    r0, r1 = power2round(i, d)
    print(f"r: {i}, r1: {r1}, r0: {r0}")

r: 0, r1: 0, r0: 0
r: 1, r1: 0, r0: 1
r: 2, r1: 0, r0: 2
r: 3, r1: 0, r0: 3
r: 4, r1: 0, r0: 4
r: 5, r1: 1, r0: -3
r: 6, r1: 1, r0: -2
r: 7, r1: 1, r0: -1
r: 8, r1: 1, r0: 0
r: 9, r1: 1, r0: 1
r: 10, r1: 1, r0: 2
r: 11, r1: 1, r0: 3
r: 12, r1: 1, r0: 4
r: 13, r1: 2, r0: -3
r: 14, r1: 2, r0: -2
r: 15, r1: 2, r0: -1
r: 16, r1: 2, r0: 0
r: 17, r1: 2, r0: 1
r: 18, r1: 2, r0: 2
r: 19, r1: 2, r0: 3
r: 20, r1: 2, r0: 4
r: 21, r1: 3, r0: -3
r: 22, r1: 3, r0: -2


In [74]:
def polynomialPower2round(t, d):
    t0 = [power2round(coefficient, d)[0] for coefficient in t]
    t1 = [power2round(coefficient, d)[1] for coefficient in t]
    return t0, t1

In [75]:
t = [2,11,5,19]
d = 3

t0, t1 = polynomialPower2round(t, d)
print(f"t: {t}, t1: {t1}, t0: {t0}")


t: [2, 11, 5, 19], t1: [0, 1, 1, 2], t0: [2, 3, -3, 3]


In [76]:
def decompose_new(r, alpha, q):
    r0 = mod_s(r, alpha)

    if r-r0 == q-1:
        r1 = 0
        r0 = r0-1
    else:
        r1 = (r - r0 + (alpha//2)) // alpha
        
    return r0, r1

def LowBits_new(r, alpha, q):
    return decompose_new(r, alpha, q)[0]

def HighBits_new(r, alpha, q):
    return decompose_new(r, alpha, q)[1]

In [77]:
def make_hint(z, r, alpha, q):
    if HighBits_new(r+z, alpha, q) != HighBits_new(r, alpha, q):
        h = 1
    else:
        h = 0
    return h

def use_hint(h, r, alpha, q, m):
    r1, r0 = decompose_new(r, alpha, q)

    if h == 1 and r0 > 0:
        r1 = (r1+1) % m
    if h == 1 and r0 < 0:
        r1 = (r1-1) % m

    return r1

## V3d - Dilithium full scheme

- ML-DSA-87 domain parameters:

    | parameter | value   |
    |-----------|---------|
    |   n       | 256     |
    |   q       | 8380417 |
    |   z       | 1753    |
    |   d       | 13      |
    |   tau     | 60      |
    |   lamb    | 256     |
    |   gamma1  | 2**19   |
    |   gamma2  | (q-1)/32|
    |   k       | 8       |
    |   l       | 7       |
    |   eta     | 2       |
    |   beta    | 120     |
    |   omega   | 75      |
    |   alpha   | 2*gamma2|
-----------------------------

- Dilithium key generation - Alice does:
    1. select random seed 256bit Ksi
    2. compute (256bit ro, 512bit ro', 256bit K) = H(Ksi, 1024)
    3. compute k by l matrix A = ExpandA(ro)
    4. compute (l length s1, k lengths s2) = ExpandS(ro'), vectors of coefficients in [-eta, eta]
    5. compute t = As1 + s2
    6. compute (t1, t0) = Power2Round(t, d)
    7. compute tr = H(ro||t1, 512)
    8. verification key is PK = (ro, t1) and signature key is SK = (ro, K, tr, s1, s2, t0)

- Dilithium signature generation - to sign message (bitstring) M, XXX does:
    1. compute k by l matrix A = ExpandA(ro)
    2. compute 512bit message representative miu = H(tr||M, 512)
    3. compute 512bit ro'' = H(K||rnd||miu, 512) with rnd is 256bit 0-string or random
    4. set kappa = 0
    5. set Found = false
    6. While Fould = false do:
        
        a. compute l length vector of polynomials y = ExpandMask(ro'', kappa) with coefficients in (-gamma1, gamma1]
        
        b. compute w = Ay and w1 = HighBits(w, 2*gamma2)
        
        c. compute c_tilda = H(miu||w1, 2*lamb)
        
        d. compute c = SampleInBall(c_tilda) and z = y + cs1
        
        e. compute r0 = LowBits(w - cs2, 2*gamma2)
        
        f. if size(z) < gamma1 - beta AND size(r0) < gamma2 - beta then

        if size(-ct0) < gamma2 then 
            compute h = MakeHint(-ct0, w-cs2+ct0, 2*gamma2)

        if number of 1's in h is <= omega then
            set Found = true
        
        g. kappa = kappa + l

    7. signature sigma = (c_tilda, z, h)

- Dilithium signature verification - to verify XXX's signature sigma = (c_tilda, z, h) on M, YYY does:
    1. obtain a copy of public key PK = (ro, t1)
    2. check that size(z) < gamma1 - beta AND that number of 1's in h <= omega (if not then reject)
    3. compute k by l matrix A = ExpandA(ro)
    4. compute compute tr = H(ro||t1, 512) and miu = H(tr||M, 512)
    5. compute c = SampleInBall(c_tilda)
    6. compute w1' = UseHint(h, Az-ct*(2^d), 2*gamma2)
    7. check that c_tilda = H(miu||w1', 2*lamb) (if not then reject)
    8. accept the signature


In [78]:
n = 256
q = 2**23 - 2**13 + 1
z = 1753
d = 13
tau = 60
lamb = 256
gamma1 = 2**19
gamma2 = (q-1)//32
k = 8
l = 7
eta = 2
beta = 120
omega = 75
alpha = 2*gamma2

# V4

## V4a - Dilithium NTT

In [79]:
def brv(v:int, n:int) -> int:
    bits_in_n = n.bit_length() - 1
    binary_string = format(v, f'0{bits_in_n}b')
    reversed_binary_string = binary_string[::-1]
    reversed_v = int(reversed_binary_string, 2)
    return reversed_v

In [80]:
brv(143, 256)

241

In [81]:
n = 8
for i in range(n//2):
    print(f"i: {i}, brv: {brv(n//2 + i, n)}")

i: 0, brv: 1
i: 1, brv: 5
i: 2, brv: 3
i: 3, brv: 7


In [82]:
def generate_zetas_array(n, z, q):
    zetas = [0] * n
    for i in range(1, n):
        zetas[i] = (z**brv(i, n))%q
    return zetas

In [83]:
q = 2**23 - 2**13 + 1
n = 256
z = 1753

zetas = generate_zetas_array(n, z, q)
print(zetas[:3])
print(zetas[-3:])

[0, 4808194, 3765607]
[7598542, 1054478, 7648983]


In [84]:
def computeNTT(f, q, n, zetas):
    w_hat = f
    m = 0
    length = n//2
    while length >= 1:
        start = 0
        while start < n:
            m = m+1
            z = zetas[m]
            for j in range(start, start+length):
                t = (z * w_hat[j+length]) % q
                w_hat[j+length] = (w_hat[j] - t) % q
                w_hat[j] = (w_hat[j] + t) % q
            start = start + 2*length
        length = length // 2
    return w_hat

In [85]:
def computeNTT_inverse(w_hat, q, n, n_inv, zetas):
    w = w_hat
    m = n
    length = 1
    while length < n:
        start = 0
        while start < n:
            m = m-1
            z = -zetas[m]
            for j in range(start, start+length):
                t = w[j]
                w[j] = (t + w[j+length]) % q
                w[j+length] = (t - w[j+length]) % q
                w[j+length] = (z * w[j+length]) % q
            start = start + 2*length
        length = 2*length

    f = n_inv
    for i in range(n):
        w[i] = (f * w[i]) % q

    return w

In [86]:
def multiplyNTT(f, g, q):
    return [(f[i] * g[i]) % q for i in range(len(f))]

In [87]:
def inverse_mod(n, q):
    return pow(n, -1, q)

In [88]:
q = 113
n = 8
z = 42

zetas = generate_zetas_array(n, z, q)
n_inv = inverse_mod(n, q)

a = [99,52,0,7,0,29,33,100]
b = [6,0,45,12,14,78,65,112]

a_hat = computeNTT(a, q, n, zetas)
b_hat = computeNTT(b, q, n, zetas)

print(f"NTT(a) = {a_hat}")
print(f"NTT(b) = {b_hat}")

c_hat = multiplyNTT(a_hat, b_hat, q)
print(f"c_hat = {c_hat}")

c = computeNTT_inverse(c_hat, q, n, n_inv, zetas)
print(f"c = {c}")

NTT(a) = [78, 65, 59, 81, 59, 60, 62, 102]
NTT(b) = [76, 92, 18, 0, 20, 105, 29, 47]
c_hat = [52, 104, 45, 0, 50, 85, 103, 48]
c = [75, 92, 86, 84, 4, 99, 25, 86]


## V4b - Kyber NTT

In [89]:
def generate_zetas_for_NTT_array(n, z, q):
    zetas = [0] * n
    
    for i in range(n):
        zetas[i] = (z**brv(i, n))%q
    return zetas

def generate_zetas_for_multiply_array(n, z, q):
    zetas = [0] * n
    
    for i in range(n):
        zetas[i] = (z**(2*brv(i, n) + 1))%q
    return zetas

In [90]:
def computeNTT(f, q, n, zetas):
    f_hat = f
    i = 1
    length = n//2

    while length >= 2:
        start = 0
        while start < n:
            zeta = zetas[i]
            i = i+1
            for j in range(start, start+length):
                t = (zeta * f_hat[j+length]) % q
                f_hat[j+length] = (f_hat[j] - t) % q
                f_hat[j] = (f_hat[j] + t) % q
            start = start + 2*length
        length = length // 2
    
    return f_hat

In [91]:
def computeNTT_inverse(f_hat, q, n, n_inv, zetas):
    f = f_hat
    i = n//2 - 1
    length = 2
    while length <= n//2:
        start = 0
        while start < n:
            zeta = zetas[i]
            i = i-1
            for j in range(start, start+length):
                t = f[j]
                f[j] = (t + f[j+length]) % q
                f[j+length] = (zeta * (f[j+length] - t)) % q
            start = start + 2*length
        length = 2*length

    f = [(coefficient * n_inv) % q for coefficient in f]

    return f

In [92]:
def base_case_multiply(a0, a1, b0, b1, q, z):
    c0 = (a0*b0 + a1*b1*z) % q
    c1 = (a0*b1 + a1*b0) % q
    return c0, c1

def multiplyNTT(f_hat, g_hat, q, n, zetas):
    h = [0] * n
    for i in range(n//2):
        a0 = f_hat[2*i]
        a1 = f_hat[2*i + 1]
        b0 = g_hat[2*i]
        b1 = g_hat[2*i + 1]
        h[2*i], h[2*i + 1] = base_case_multiply(a0, a1, b0, b1, q, zetas[i])

    return h

In [93]:
q = 3329
n = 256
z = 17

n_inv = inverse_mod(n//2, q)
print(f"n_inv: {n_inv}")
print()
zetas_for_NTT = generate_zetas_for_NTT_array(n//2, z, q)
print(zetas_for_NTT[:3])
print(zetas_for_NTT[-3:])
print()
zetas_for_multiply = generate_zetas_for_multiply_array(n//2, z, q)
print(zetas_for_multiply[:3])
print(zetas_for_multiply[-3:])

n_inv: 3303

[1, 1729, 2580]
[2935, 885, 2154]

[17, 3312, 2761]
[2444, 2154, 1175]


In [94]:
q = 137
n = 8
n_inv = inverse_mod(n//2, q)
z = 10
zetas_for_NTT = generate_zetas_for_NTT_array(n//2, z, q)
zetas_for_multiply = generate_zetas_for_multiply_array(n//2, z, q)

a = [99,52,0,7,0,29,33,100]
b = [6,0,45,12,14,78,65,112]

a_hat = computeNTT(a, q, n, zetas_for_NTT)
b_hat = computeNTT(b, q, n, zetas_for_NTT)

print(f"NTT(a) = {a_hat}")
print(f"NTT(b) = {b_hat}")

c_hat = multiplyNTT(a_hat, b_hat, q, n, zetas_for_multiply)
print(f"c_hat = {c_hat}")

c = computeNTT_inverse(c_hat, q, n, n_inv, zetas_for_NTT)
print(f"c = {c}")

NTT(a) = [82, 135, 116, 15, 18, 83, 43, 112]
NTT(b) = [0, 45, 72, 74, 5, 114, 84, 41]
c_hat = [59, 128, 129, 74, 48, 1, 16, 74]
c = [63, 35, 21, 40, 51, 113, 121, 31]
